In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations

import pandas as pd
import numpy as np

from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

cwd = Path.cwd()
BASE_DIR = cwd.parent if cwd.name == "notebooks" else cwd

PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

news_all = pd.read_csv(PROCESSED_DIR / "news_all.csv")
train_samples = pd.read_csv(PROCESSED_DIR / "train_samples_5000.csv")
dev_samples = pd.read_csv(PROCESSED_DIR / "dev_samples_5000.csv")

news_all["text"] = news_all["text"].fillna("")
train_samples["history"] = train_samples["history"].fillna("")
dev_samples["history"] = dev_samples["history"].fillna("")

print("news_all:", news_all.shape)
print("train_samples:", train_samples.shape)
print("dev_samples:", dev_samples.shape)

news_all: (65238, 6)
train_samples: (187715, 5)
dev_samples: (187009, 5)


In [ ]:
news_id_to_category = dict(zip(news_all["news_id"], news_all["category"]))
news_id_to_subcategory = dict(zip(news_all["news_id"], news_all["subcategory"]))

news_id_to_text = dict(zip(news_all["news_id"], news_all["text"]))

news_all.head()

,news_id,category,subcategory,title,abstract,text
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...","The Brands Queen Elizabeth, Prince Charles, an..."
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,50 Worst Habits For Belly Fat These seemingly ...
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,The Cost of Trump's Aid Freeze in the Trenches...
3,N53526,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",I Was An NBA Wife. Here's How It Affected My M...
4,N38324,health,medical,"How to Get Rid of Skin Tags, According to a De...","They seem harmless, but there's a very good re...","How to Get Rid of Skin Tags, According to a De..."


In [ ]:
vectorizer = TfidfVectorizer(
    max_features=50000,
    stop_words="english"
)

news_tfidf = vectorizer.fit_transform(news_all["text"])

news_id_to_idx = {
    news_id: idx
    for idx, news_id in enumerate(news_all["news_id"])
}

print(news_tfidf.shape)

(65238, 50000)


In [ ]:
def parse_history(history, max_history=50):
    if pd.isna(history) or history == "":
        return []
    return str(history).split()[-max_history:]


def build_user_click_sets(samples, max_clicks_per_user=50):
    user_clicks = {}

    for user_id, group in tqdm(samples.groupby("user_id"), desc="building user click sets"):
        clicks = []

        # history에 있는 과거 클릭 뉴스들
        for hist in group["history"].dropna().unique():
            clicks.extend(str(hist).split())

        # impression에서 실제 클릭한 후보 뉴스들
        positive_clicks = group.loc[group["label"] == 1, "candidate_news"].tolist()
        clicks.extend(positive_clicks)

        # 순서 유지하며 중복 제거
        seen = set()
        unique_clicks = []
        for nid in clicks:
            if nid not in seen:
                unique_clicks.append(nid)
                seen.add(nid)

        # 너무 긴 history는 최근 일부만 사용
        user_clicks[user_id] = set(unique_clicks[-max_clicks_per_user:])

    return user_clicks


user_clicks = build_user_click_sets(train_samples, max_clicks_per_user=50)

print("num users:", len(user_clicks))

building user click sets:   0%|          | 0/4640 [00:00<?, ?it/s]

num users: 4640


In [ ]:
def build_npmi_dict(user_clicks, min_pair_count=2):
    item_counts = Counter()
    pair_counts = Counter()

    users = list(user_clicks.keys())
    n_users = len(users)

    for user in tqdm(users, desc="counting item pairs"):
        items = sorted(list(user_clicks[user]))

        item_counts.update(items)

        for a, b in combinations(items, 2):
            pair_counts[(a, b)] += 1

    npmi_dict = {}

    for (a, b), cij in tqdm(pair_counts.items(), desc="computing NPMI"):
        if cij < min_pair_count:
            continue

        ci = item_counts[a]
        cj = item_counts[b]

        p_i = ci / n_users
        p_j = cj / n_users
        p_ij = cij / n_users

        if p_ij <= 0 or p_i <= 0 or p_j <= 0 or p_ij >= 1:
            continue

        pmi = np.log(p_ij / (p_i * p_j))
        npmi = pmi / (-np.log(p_ij))

        # 추천 feature로 쓰기 위해 음수 관계는 0으로 clip
        npmi = max(0.0, float(npmi))

        npmi_dict[(a, b)] = npmi
        npmi_dict[(b, a)] = npmi

    return npmi_dict, item_counts


npmi_dict, item_counts = build_npmi_dict(user_clicks, min_pair_count=2)

print("NPMI pairs:", len(npmi_dict))
print("items:", len(item_counts))

counting item pairs:   0%|          | 0/4640 [00:00<?, ?it/s]

computing NPMI:   0%|          | 0/1514522 [00:00<?, ?it/s]

NPMI pairs: 393326
items: 15648


In [ ]:
click_counts = train_samples[train_samples["label"] == 1].groupby("candidate_news").size()

max_click = click_counts.max()

popularity_score = {
    news_id: np.log1p(count) / np.log1p(max_click)
    for news_id, count in click_counts.items()
}

print("num popular items:", len(popularity_score))

num popular items: 1924


In [ ]:
def max_npmi_score(history_ids, candidate_news):
    scores = [
        npmi_dict.get((hist_news, candidate_news), 0.0)
        for hist_news in history_ids
    ]
    return max(scores) if scores else 0.0


def make_feature_table(samples, name="samples", max_history=50):
    rows = []

    grouped = samples.groupby("impression_id")

    for impression_id, group in tqdm(grouped, desc=f"making features for {name}"):
        history = group["history"].iloc[0]
        history_ids = parse_history(history, max_history=max_history)

        # category preference
        hist_categories = [
            news_id_to_category.get(nid)
            for nid in history_ids
            if nid in news_id_to_category
        ]
        hist_categories = [x for x in hist_categories if pd.notna(x)]
        category_counter = Counter(hist_categories)
        category_total = len(hist_categories)

        # subcategory preference
        hist_subcategories = [
            news_id_to_subcategory.get(nid)
            for nid in history_ids
            if nid in news_id_to_subcategory
        ]
        hist_subcategories = [x for x in hist_subcategories if pd.notna(x)]
        subcategory_counter = Counter(hist_subcategories)
        subcategory_total = len(hist_subcategories)

        # TF-IDF user vector
        hist_indices = [
            news_id_to_idx[nid]
            for nid in history_ids
            if nid in news_id_to_idx
        ]

        if len(hist_indices) > 0:
            user_vec = news_tfidf[hist_indices].mean(axis=0)
        else:
            user_vec = None

        for _, row in group.iterrows():
            candidate = row["candidate_news"]

            # 1. NPMI feature
            npmi_score = max_npmi_score(history_ids, candidate)

            # 2. category preference
            cand_category = news_id_to_category.get(candidate)
            if category_total > 0 and pd.notna(cand_category):
                category_pref = category_counter.get(cand_category, 0) / category_total
            else:
                category_pref = 0.0

            # 3. subcategory preference
            cand_subcategory = news_id_to_subcategory.get(candidate)
            if subcategory_total > 0 and pd.notna(cand_subcategory):
                subcategory_pref = subcategory_counter.get(cand_subcategory, 0) / subcategory_total
            else:
                subcategory_pref = 0.0

            # 4. TF-IDF similarity
            if user_vec is not None and candidate in news_id_to_idx:
                cand_vec = news_tfidf[news_id_to_idx[candidate]]
                tfidf_score = float(user_vec @ cand_vec.T)
            else:
                tfidf_score = 0.0

            # 5. popularity proxy
            pop_score = popularity_score.get(candidate, 0.0)

            rows.append({
                "impression_id": row["impression_id"],
                "user_id": row["user_id"],
                "history": row["history"],
                "candidate_news": candidate,
                "label": row["label"],
                "npmi_score": npmi_score,
                "category_pref": category_pref,
                "subcategory_pref": subcategory_pref,
                "tfidf_score": tfidf_score,
                "popularity_score": pop_score
            })

    return pd.DataFrame(rows)

In [ ]:
train_features = make_feature_table(train_samples, name="train")
dev_features = make_feature_table(dev_samples, name="dev")

print(train_features.shape)
print(dev_features.shape)

train_features.head()

making features for train:   0%|          | 0/5000 [00:00<?, ?it/s]

C:\Users\rlagu\AppData\Local\Temp\ipykernel_16732\587165830.py:73: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  tfidf_score = float(user_vec @ cand_vec.T)


making features for dev:   0%|          | 0/5000 [00:00<?, ?it/s]

(187715, 10)
(187009, 10)


,impression_id,user_id,history,candidate_news,label,npmi_score,category_pref,subcategory_pref,tfidf_score,popularity_score
0,1,U13740,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N55689,1,0.253139,0.222222,0.0,0.011592,1.000000
1,1,U13740,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N35729,0,0.059543,0.333333,0.0,0.002918,0.957276
2,2,U91836,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N20678,0,0.231109,0.020000,0.0,0.001734,0.556342
3,2,U91836,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N39317,0,0.000000,0.620000,0.0,0.002054,0.601117
4,2,U91836,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N58114,0,0.000000,0.000000,0.0,0.004919,0.390462


In [ ]:
train_features.to_csv(
    PROCESSED_DIR / "train_airs_lite_features.csv",
    index=False,
    encoding="utf-8-sig"
)

dev_features.to_csv(
    PROCESSED_DIR / "dev_airs_lite_features.csv",
    index=False,
    encoding="utf-8-sig"
)

print("saved feature tables")

saved feature tables


In [ ]:
feature_cols = [
    "npmi_score",
    "category_pref",
    "subcategory_pref",
    "tfidf_score",
    "popularity_score"
]

X_train = train_features[feature_cols].fillna(0)
y_train = train_features["label"].astype(int)

X_dev = dev_features[feature_cols].fillna(0)

lr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

lr_model.fit(X_train, y_train)

dev_features["score"] = lr_model.predict_proba(X_dev)[:, 1]

dev_features.head()

,impression_id,user_id,history,candidate_news,label,npmi_score,category_pref,subcategory_pref,tfidf_score,popularity_score,score
0,1,U80234,N55189 N46039 N51741 N53234 N11276 N264 N40716...,N28682,0,0.0,0.000000,0.000000,0.003380,0.000000,0.136488
1,1,U80234,N55189 N46039 N51741 N53234 N11276 N264 N40716...,N48740,0,0.0,0.066667,0.000000,0.008121,0.000000,0.152094
2,1,U80234,N55189 N46039 N51741 N53234 N11276 N264 N40716...,N31958,1,0.0,0.000000,0.000000,0.005173,0.000000,0.139039
3,1,U80234,N55189 N46039 N51741 N53234 N11276 N264 N40716...,N34130,0,0.0,0.266667,0.133333,0.001394,0.000000,0.210521
4,1,U80234,N55189 N46039 N51741 N53234 N11276 N264 N40716...,N6916,0,0.0,0.066667,0.000000,0.003035,0.139085,0.182080


In [ ]:
coef = lr_model.named_steps["lr"].coef_[0]

coef_table = pd.DataFrame({
    "feature": feature_cols,
    "coef": coef
}).sort_values("coef", ascending=False)

coef_table

,feature,coef
0,npmi_score,0.688051
4,popularity_score,0.461446
1,category_pref,0.201999
2,subcategory_pref,0.165514
3,tfidf_score,0.119395


In [ ]:
def mrr_score(labels, scores):
    labels = np.array(labels)
    scores = np.array(scores)

    order = np.argsort(scores)[::-1]
    sorted_labels = labels[order]

    positive_indices = np.where(sorted_labels == 1)[0]

    if len(positive_indices) == 0:
        return np.nan

    return 1.0 / (positive_indices[0] + 1)


def dcg_at_k(labels, scores, k):
    labels = np.array(labels)
    scores = np.array(scores)

    order = np.argsort(scores)[::-1]
    sorted_labels = labels[order][:k]

    gains = sorted_labels
    discounts = np.log2(np.arange(2, len(sorted_labels) + 2))

    return np.sum(gains / discounts)


def ndcg_at_k(labels, scores, k):
    dcg = dcg_at_k(labels, scores, k)

    ideal_scores = np.array(labels)
    idcg = dcg_at_k(labels, ideal_scores, k)

    if idcg == 0:
        return np.nan

    return dcg / idcg


def evaluate_by_impression(scored_df):
    metrics = []

    for impression_id, group in scored_df.groupby("impression_id"):
        labels = group["label"].values
        scores = group["score"].values

        if len(np.unique(labels)) < 2:
            auc = np.nan
        else:
            auc = roc_auc_score(labels, scores)

        metrics.append({
            "impression_id": impression_id,
            "AUC": auc,
            "MRR": mrr_score(labels, scores),
            "nDCG@5": ndcg_at_k(labels, scores, k=5),
            "nDCG@10": ndcg_at_k(labels, scores, k=10)
        })

    metrics_df = pd.DataFrame(metrics)
    result = metrics_df[["AUC", "MRR", "nDCG@5", "nDCG@10"]].mean(skipna=True)

    return result, metrics_df

In [ ]:
airs_lite_result, airs_lite_metrics_df = evaluate_by_impression(dev_features)

airs_lite_result

AUC        0.617982
MRR        0.320763
nDCG@5     0.305846
nDCG@10    0.369925
dtype: float64

In [ ]:
airs_lite_result_table = pd.DataFrame([{
    "model": "AiRS-lite",
    "AUC": airs_lite_result["AUC"],
    "MRR": airs_lite_result["MRR"],
    "nDCG@5": airs_lite_result["nDCG@5"],
    "nDCG@10": airs_lite_result["nDCG@10"]
}])

airs_lite_result_table

,model,AUC,MRR,nDCG@5,nDCG@10
0,AiRS-lite,0.617982,0.320763,0.305846,0.369925


In [ ]:
results_path = OUTPUT_DIR / "results.csv"

if results_path.exists():
    results = pd.read_csv(results_path)
else:
    results = pd.DataFrame(columns=["model", "AUC", "MRR", "nDCG@5", "nDCG@10"])

# 기존 AiRS-lite 결과가 있으면 제거 후 갱신
results = results[results["model"] != "AiRS-lite"]

results = pd.concat([results, airs_lite_result_table], ignore_index=True)

results.to_csv(results_path, index=False, encoding="utf-8-sig")

dev_features.to_csv(
    OUTPUT_DIR / "airs_lite_dev_scored.csv",
    index=False,
    encoding="utf-8-sig"
)

airs_lite_metrics_df.to_csv(
    OUTPUT_DIR / "airs_lite_metrics_by_impression.csv",
    index=False,
    encoding="utf-8-sig"
)

results

,model,AUC,MRR,nDCG@5,nDCG@10
0,TF-IDF baseline,0.577705,0.322725,0.297889,0.361070
1,AiRS-lite,0.617982,0.320763,0.305846,0.369925


In [ ]:
coef_table

,feature,coef
0,npmi_score,0.688051
4,popularity_score,0.461446
1,category_pref,0.201999
2,subcategory_pref,0.165514
3,tfidf_score,0.119395


In [ ]:
def train_and_evaluate_with_features(selected_features, model_name):
    X_train = train_features[selected_features].fillna(0)
    y_train = train_features["label"].astype(int)

    X_dev = dev_features[selected_features].fillna(0)

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    scored_dev = dev_features.copy()
    scored_dev["score"] = model.predict_proba(X_dev)[:, 1]

    result, metrics_df = evaluate_by_impression(scored_dev)

    return {
        "model": model_name,
        "AUC": result["AUC"],
        "MRR": result["MRR"],
        "nDCG@5": result["nDCG@5"],
        "nDCG@10": result["nDCG@10"]
    }

In [ ]:
all_features = [
    "npmi_score",
    "category_pref",
    "subcategory_pref",
    "tfidf_score",
    "popularity_score"
]

ablation_settings = {
    "AiRS-lite full": all_features,

    "w/o NPMI": [
        "category_pref",
        "subcategory_pref",
        "tfidf_score",
        "popularity_score"
    ],

    "w/o popularity": [
        "npmi_score",
        "category_pref",
        "subcategory_pref",
        "tfidf_score"
    ],

    "w/o category features": [
        "npmi_score",
        "tfidf_score",
        "popularity_score"
    ],

    "w/o TF-IDF": [
        "npmi_score",
        "category_pref",
        "subcategory_pref",
        "popularity_score"
    ],

    "only TF-IDF": [
        "tfidf_score"
    ],

    "only NPMI": [
        "npmi_score"
    ],

    "only popularity": [
        "popularity_score"
    ]
}

ablation_results = []

for model_name, features in ablation_settings.items():
    print("running:", model_name, features)
    result = train_and_evaluate_with_features(features, model_name)
    ablation_results.append(result)

ablation_df = pd.DataFrame(ablation_results)

ablation_df

In [ ]:
baseline_result = pd.read_csv(OUTPUT_DIR / "results.csv")

comparison_df = pd.concat([
    baseline_result,
    ablation_df
], ignore_index=True)

comparison_df

In [ ]:
ablation_df.to_csv(
    OUTPUT_DIR / "airs_lite_ablation.csv",
    index=False,
    encoding="utf-8-sig"
)

comparison_df.to_csv(
    OUTPUT_DIR / "results_with_ablation.csv",
    index=False,
    encoding="utf-8-sig"
)

print("saved!")